# Hierarchical HMM allocation

Nested hidden Markov models estimate broad and fine market regimes from volatility and semivariance. The estimated state controls the portfolio construction method.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
REGIME-CONDITIONAL PORTFOLIO STRATEGIES WITH HAR-SEMIVARIANCE AND HIERARCHICAL HMM
==================================================================================

HIERARCHICAL HMM VERSION - CONVERGENCE FIXES APPLIED
====================================================

CRITICAL FIXES FOR CONVERGENCE:
1. Enhanced data validation and outlier removal
2. Increased minimum covariance for numerical stability
3. Robust feature scaling with outlier clipping
4. Better handling of insufficient regime-specific data
5. Improved initialization with regularization
6. More lenient convergence criteria for sub-HMMs
7. Fallback mechanisms for edge cases

NO CHANGES TO FEATURES OR MODEL STRUCTURE - ONLY CONVERGENCE IMPROVEMENTS
"""

from _market_data import download_ohlcv

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

try:
    from hmmlearn import hmm
except ImportError:
    print("ERROR: hmmlearn not installed. Install with: pip install hmmlearn")
    raise

# ============================================================================
# CONFIGURATION PARAMETERS (FULLY CONFIGURABLE)
# ============================================================================


# Date range for backtest
REGIME_TICKER = 'SPY'
BENCHMARK_TICKER = 'URTH'
START_DATE = '2013-01-01'
END_DATE = '2018-12-31'

# Asset universe - fully configurable
GROWTH_ASSETS = ['SPY', 'QQQ', 'IWM', 'VTI', 'XLK', 'XLV', 'XLF', 'XLE', 'AAPL', 'MSFT']
DEFENSIVE_ASSETS = ['TLT', 'GLD', 'UUP']
SHORT_ASSETS = ['SH']
ALL_ASSETS = list(set(GROWTH_ASSETS + DEFENSIVE_ASSETS + SHORT_ASSETS))

# Feature engineering parameters
DAILY_LAG = 1
WEEKLY_LAG = 5
MONTHLY_LAG = 22
USE_VIX = True

# Regime detection parameters
N_TOP_REGIMES = 3
N_SUB_REGIMES = 2
REGIME_REBALANCE_FREQ = 21
MIN_TRAIN_DAYS = 252

# ============================================================================
# CONVERGENCE FIX #1: IMPROVED HMM PARAMETERS FOR STABILITY
# ============================================================================
# Increase minimum covariance to prevent singular matrices
# This is CRITICAL for convergence - too small values cause numerical issues
HMM_TOP_MIN_COVAR = 1e-3  # INCREASED from 1e-4 (10x more regularization)
HMM_SUB_MIN_COVAR = 1e-3  # INCREASED from 1e-4

# More iterations and lenient tolerance for better convergence
HMM_TOP_N_ITER = 300  # INCREASED from 200 (50% more iterations)
HMM_TOP_TOL = 5e-3    # RELAXED from 1e-3 (5x more lenient)
HMM_TOP_N_INIT = 10   # INCREASED from 5 (more initialization attempts)

# Sub-HMM parameters - even more lenient due to less data
HMM_SUB_N_ITER = 250  # INCREASED from 150
HMM_SUB_TOL = 1e-2    # RELAXED from 1e-3 (10x more lenient)
HMM_SUB_N_INIT = 8    # INCREASED from 3

# Covariance type remains same
HMM_TOP_COVARIANCE_TYPE = 'diag'
HMM_SUB_COVARIANCE_TYPE = 'diag'
HMM_TOP_RANDOM_STATE = 42
HMM_SUB_RANDOM_STATE = 123

# ============================================================================
# CONVERGENCE FIX #2: MINIMUM DATA REQUIREMENTS
# ============================================================================
# Increase minimum data requirements to ensure sufficient training data
MIN_SUB_REGIME_DAYS = 90  # INCREASED from 60 (need more data for sub-HMMs)
MIN_FEATURES_VALID_PCT = 0.90  # Require 90% valid features (no NaNs)

# Portfolio construction parameters
MAX_HRP_ASSETS = 10
TRANSACTION_COST = 0.001
INITIAL_CAPITAL = 100000

# Risk Parity parameters
MIN_RISK_PARITY_WEIGHT = 0.05
MAX_RISK_PARITY_WEIGHT = 0.40

# ============================================================================
# DATA ACQUISITION (UNCHANGED)
# ============================================================================

def fetch_price_data(ticker, start_date, end_date):
    try:
        frame = download_ohlcv(ticker.replace("%5E", "^"), start_date, end_date)
    except RuntimeError:
        return None
    return frame.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]


def fetch_all_assets(tickers, start_date, end_date):
    """
    Fetch all asset data from Yahoo Finance.
    Only includes tickers with sufficient data (>200 days).
    NO BACKFILL OR FORWARD FILL.
    """
    print(f"\n{'='*80}")
    print("FETCHING ASSET DATA")
    print(f"{'='*80}")
    
    asset_data = {}
    for ticker in tickers:
        try:
            print(f"  Fetching {ticker}...", end='')
            df = fetch_price_data(ticker, start_date, end_date)
            if df is not None and len(df) > 200:
                asset_data[ticker] = df
                print(f" ✓ {len(df)} days")
            else:
                print(f" ✗ Insufficient data")
        except Exception as e:
            print(f" ✗ Error: {str(e)}")
    
    print(f"\n✓ Successfully fetched {len(asset_data)} assets")
    return asset_data


# ============================================================================
# HAR-SEMIVARIANCE FEATURES (UNCHANGED)
# ============================================================================

def calculate_garman_klass_vol(prices):
    """
    Calculate Garman-Klass volatility estimator.
    Returns annualized volatility in percentage terms.
    """
    log_hl = np.log(prices['high'] / prices['low'])
    log_co = np.log(prices['close'] / prices['open'])
    gk_var = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)
    
    rv_series = pd.Series(np.sqrt(gk_var.clip(lower=0) * 252) * 100, index=prices.index)
    return rv_series


def prepare_regime_features(spy_data, vix_data=None):
    """
    Prepare HAR-semivariance features for HIERARCHICAL regime detection.
    
    UNCHANGED: Same features as before
    Creates TWO feature sets:
    1. VOLATILITY FEATURES (for top-level macro regime HMM)
    2. RETURN FEATURES (for bottom-level sub-regime HMMs)
    
    ALL FEATURES ARE LAGGED TO AVOID LOOK-AHEAD BIAS.
    """
    
    print(f"\n{'='*80}")
    print("PREPARING HIERARCHICAL REGIME DETECTION FEATURES")
    print(f"{'='*80}")
    
    # FEATURE SET 1: VOLATILITY FEATURES (for top-level HMM)
    vol_features = pd.DataFrame(index=spy_data.index)
    
    rv = calculate_garman_klass_vol(spy_data)
    returns = np.log(spy_data['close'] / spy_data['close'].shift(1))
    
    gk_var = (rv / (np.sqrt(252) * 100)) ** 2
    rs_pos_array = np.where(returns > 0, gk_var, 0)
    rs_neg_array = np.where(returns <= 0, gk_var, 0)
    
    rs_pos = pd.Series(np.sqrt(rs_pos_array * 252) * 100, index=spy_data.index)
    rs_neg = pd.Series(np.sqrt(rs_neg_array * 252) * 100, index=spy_data.index)
    
    # HAR volatility components - lagged by 1 day
    vol_features['RV_D'] = rv.shift(DAILY_LAG)
    vol_features['RV_W'] = rv.shift(1).rolling(WEEKLY_LAG).mean()
    vol_features['RV_M'] = rv.shift(1).rolling(MONTHLY_LAG).mean()
    vol_features['RS_neg_D'] = rs_neg.shift(DAILY_LAG)
    vol_features['RS_neg_W'] = rs_neg.shift(1).rolling(WEEKLY_LAG).mean()
    
    # VIX features (if available)
    if vix_data is not None:
        vix_aligned = vix_data.reindex(spy_data.index)
        vol_features['VIX_level_D'] = vix_aligned.shift(DAILY_LAG)
        vol_features['VIX_level_W'] = vix_aligned.shift(1).rolling(WEEKLY_LAG).mean()
    
    # Asymmetry ratio
    rs_pos_shifted = rs_pos.shift(DAILY_LAG)
    vol_features['Asym_ratio_D'] = vol_features['RS_neg_D'] / (rs_pos_shifted + 1e-8)
    
    # FEATURE SET 2: RETURN FEATURES (for sub-level HMMs)
    ret_features = pd.DataFrame(index=spy_data.index)
    
    ret_features['Ret_D'] = returns.shift(DAILY_LAG)
    ret_features['Ret_W'] = returns.shift(1).rolling(WEEKLY_LAG).mean()
    ret_features['Ret_M'] = returns.shift(1).rolling(MONTHLY_LAG).mean()
    
    ret_features['Mom_W'] = (spy_data['close'] / spy_data['close'].shift(WEEKLY_LAG) - 1).shift(1)
    ret_features['Mom_M'] = (spy_data['close'] / spy_data['close'].shift(MONTHLY_LAG) - 1).shift(1)
    
    ret_features['Ret_Skew_W'] = returns.shift(1).rolling(WEEKLY_LAG).skew()
    
    # Drop rows with NaN values - NO BACKFILL OR FORWARD FILL
    vol_features = vol_features.dropna()
    ret_features = ret_features.dropna()
    
    # Align both feature sets
    common_dates = vol_features.index.intersection(ret_features.index)
    vol_features = vol_features.loc[common_dates]
    ret_features = ret_features.loc[common_dates]
    
    print(f"✓ Prepared {len(vol_features)} observations")
    print(f"  VOLATILITY features: {list(vol_features.columns)}")
    print(f"  RETURN features: {list(ret_features.columns)}")
    
    return vol_features, ret_features


# ============================================================================
# CONVERGENCE FIX #3: ROBUST DATA VALIDATION AND CLEANING
# ============================================================================

def validate_and_clean_features(X, feature_names, outlier_std=4.0):
    """
    CONVERGENCE FIX: Validate and clean features for HMM fitting.
    
    CRITICAL for convergence:
    1. Remove extreme outliers (beyond outlier_std standard deviations)
    2. Check for infinite values
    3. Ensure sufficient data variation
    4. Replace problematic values with median
    
    Args:
        X: Feature matrix (numpy array)
        feature_names: List of feature names
        outlier_std: Number of standard deviations for outlier detection
        
    Returns:
        Cleaned feature matrix
    """
    X_clean = X.copy()
    
    for i, feat_name in enumerate(feature_names):
        feat_col = X_clean[:, i]
        
        # Remove infinite values
        inf_mask = np.isinf(feat_col)
        if inf_mask.any():
            print(f"    WARNING: {feat_name} has {inf_mask.sum()} infinite values - replacing with median")
            median_val = np.median(feat_col[~inf_mask])
            feat_col[inf_mask] = median_val
        
        # Remove extreme outliers (beyond outlier_std standard deviations)
        mean_val = np.mean(feat_col)
        std_val = np.std(feat_col)
        
        if std_val > 0:
            z_scores = np.abs((feat_col - mean_val) / std_val)
            outlier_mask = z_scores > outlier_std
            
            if outlier_mask.any():
                print(f"    WARNING: {feat_name} has {outlier_mask.sum()} outliers (>{outlier_std}σ) - clipping")
                lower_bound = mean_val - outlier_std * std_val
                upper_bound = mean_val + outlier_std * std_val
                feat_col = np.clip(feat_col, lower_bound, upper_bound)
        
        X_clean[:, i] = feat_col
    
    return X_clean


def validate_training_data(X, min_samples, min_features_complete):
    """
    CONVERGENCE FIX: Validate training data quality.
    
    Returns True if data meets minimum requirements, False otherwise.
    """
    # Check minimum samples
    if len(X) < min_samples:
        print(f"    INSUFFICIENT DATA: {len(X)} samples (need {min_samples})")
        return False
    
    # Check for NaN values
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        print(f"    DATA QUALITY ISSUE: {nan_count} NaN values detected")
        return False
    
    # Check feature variation (avoid constant features)
    for i in range(X.shape[1]):
        if np.std(X[:, i]) < 1e-8:
            print(f"    DATA QUALITY ISSUE: Feature {i} has no variation")
            return False
    
    return True


# ============================================================================
# CONVERGENCE FIX #4: IMPROVED HMM INITIALIZATION
# ============================================================================

def initialize_hmm_params_robust(X_train_scaled, n_regimes, random_seed, min_covar):
    """
    CONVERGENCE FIX: More robust HMM parameter initialization.
    
    IMPROVEMENTS:
    1. Better K-means initialization with more attempts
    2. Regularized covariance matrices
    3. Stronger diagonal bias in transition matrix
    4. Better handling of small clusters
    
    Args:
        X_train_scaled: Scaled training features
        n_regimes: Number of HMM states
        random_seed: Random seed for reproducibility
        min_covar: Minimum covariance value for regularization
        
    Returns:
        Dictionary with initialized HMM parameters
    """
    from sklearn.cluster import KMeans
    
    # IMPROVEMENT: More robust K-means with multiple restarts
    kmeans = KMeans(
        n_clusters=n_regimes, 
        random_state=random_seed, 
        n_init=20,  # INCREASED from 10 (more initialization attempts)
        max_iter=500  # More iterations for convergence
    )
    kmeans_labels = kmeans.fit_predict(X_train_scaled)
    
    # Initialize means from K-means centroids
    means_init = kmeans.cluster_centers_
    
    # IMPROVEMENT: More robust covariance initialization
    covars_init = np.zeros((n_regimes, X_train_scaled.shape[1]))
    
    for i in range(n_regimes):
        cluster_mask = kmeans_labels == i
        cluster_size = cluster_mask.sum()
        
        if cluster_size > 1:
            cluster_data = X_train_scaled[cluster_mask]
            cluster_var = np.var(cluster_data, axis=0)
            
            # REGULARIZATION: Ensure sufficient variance
            # Add larger minimum covariance for numerical stability
            covars_init[i] = np.maximum(cluster_var, min_covar * 10)  # 10x min_covar for safety
        else:
            # FALLBACK: Use global variance with strong regularization
            global_var = np.var(X_train_scaled, axis=0)
            covars_init[i] = np.maximum(global_var, min_covar * 10)
    
    # IMPROVEMENT: Stronger persistence bias in transition matrix
    # Regimes should be more "sticky" to reduce noise
    transmat_init = np.full((n_regimes, n_regimes), 0.02)  # Lower off-diagonal (was 0.05)
    np.fill_diagonal(transmat_init, 0.94)  # Higher diagonal (was 0.85)
    transmat_init = transmat_init / transmat_init.sum(axis=1, keepdims=True)
    
    # Uniform initial state distribution
    startprob_init = np.full(n_regimes, 1.0 / n_regimes)
    
    return {
        'startprob_': startprob_init,
        'transmat_': transmat_init,
        'means_': means_init,
        'covars_': covars_init
    }


# ============================================================================
# CONVERGENCE FIX #5: IMPROVED HMM FITTING WITH ROBUSTNESS
# ============================================================================

def fit_hmm_with_multiple_inits_robust(X_train_scaled, n_regimes, n_init_attempts, 
                                       n_iter, tol, random_seed, cov_type, min_covar,
                                       feature_names=None):
    """
    CONVERGENCE FIX: Fit HMM with enhanced robustness.
    
    IMPROVEMENTS:
    1. Data validation before fitting
    2. Outlier removal
    3. Better initialization
    4. Multiple attempts with different seeds
    5. Fallback mechanisms
    
    Returns the model with best log-likelihood.
    """
    # VALIDATION: Check data quality before fitting
    if not validate_training_data(X_train_scaled, min_samples=50, min_features_complete=0.9):
        print("    CONVERGENCE FIX: Training data validation FAILED")
        return None, -np.inf, False
    
    # CLEANING: Remove outliers for numerical stability
    if feature_names is not None:
        X_train_cleaned = validate_and_clean_features(
            X_train_scaled, 
            feature_names, 
            outlier_std=4.0  # Clip beyond 4 standard deviations
        )
    else:
        X_train_cleaned = X_train_scaled
    
    best_model = None
    best_score = -np.inf
    best_converged = False
    
    print(f"    Attempting {n_init_attempts} initializations...")
    
    for attempt in range(n_init_attempts):
        try:
            # Create model with specified parameters
            model = hmm.GaussianHMM(
                n_components=n_regimes,
                covariance_type=cov_type,
                n_iter=n_iter,
                tol=tol,
                random_state=random_seed + attempt,
                verbose=False,
                init_params='',  # Manual initialization
                params='stmc'
            )
            
            # IMPROVED INITIALIZATION
            init_params = initialize_hmm_params_robust(
                X_train_cleaned, 
                n_regimes, 
                random_seed + attempt,
                min_covar
            )
            
            model.startprob_ = init_params['startprob_']
            model.transmat_ = init_params['transmat_']
            model.means_ = init_params['means_']
            model.covars_ = init_params['covars_']
            
            # Fit the model
            model.fit(X_train_cleaned)
            
            # Compute log-likelihood
            score = model.score(X_train_cleaned)
            
            # Check convergence status
            converged = model.monitor_.converged if hasattr(model, 'monitor_') else False
            
            # Update best model
            if converged and score > best_score:
                best_score = score
                best_model = model
                best_converged = True
                print(f"      Attempt {attempt+1}: CONVERGED (score={score:.2f}) ✓")
            elif not best_converged and score > best_score:
                # Accept non-converged if no converged model found yet
                best_score = score
                best_model = model
                print(f"      Attempt {attempt+1}: NOT converged (score={score:.2f})")
                
        except Exception as e:
            print(f"      Attempt {attempt+1}: FAILED ({str(e)})")
            continue
    
    if best_model is not None:
        status = "CONVERGED" if best_converged else "BEST EFFORT"
        print(f"    Final: {status} model (score={best_score:.2f})")
    else:
        print(f"    CONVERGENCE FIX: All initialization attempts FAILED")
    
    return best_model, best_score, best_converged


# ============================================================================
# CONVERGENCE FIX #6: IMPROVED HIERARCHICAL REGIME DETECTION
# ============================================================================

def detect_hierarchical_regime_walk_forward(vol_features, ret_features, current_date, 
                                           n_top_regimes=3, n_sub_regimes=2, 
                                           min_train_days=252):
    """
    CONVERGENCE FIX: HIERARCHICAL HMM with enhanced robustness.
    
    IMPROVEMENTS:
    1. Better data validation
    2. Robust feature scaling (RobustScaler instead of StandardScaler)
    3. Stricter minimum data requirements for sub-HMMs
    4. Better handling of edge cases
    5. Fallback to top-level only if sub-HMM fails
    
    Returns:
    - top_regime, sub_regime, combined_regime, top_model, sub_model
    - Returns (None, None, None, None, None) if convergence fails
    """
    # Get training data STRICTLY before current_date
    train_mask = vol_features.index < current_date
    train_data_vol = vol_features.loc[train_mask]
    train_data_ret = ret_features.loc[train_mask]
    
    # Need sufficient training data
    if len(train_data_vol) < min_train_days:
        return None, None, None, None, None
    
    # ========================================================================
    # LEVEL 1: TOP-LEVEL HMM (Macro volatility regime)
    # ========================================================================
    try:
        X_train_vol = train_data_vol.values
        
        # CONVERGENCE FIX: Use RobustScaler instead of StandardScaler
        # RobustScaler is more resistant to outliers (uses median and IQR)
        scaler_vol = RobustScaler()
        X_train_vol_scaled = scaler_vol.fit_transform(X_train_vol)
        
        print(f"\n  TOP-LEVEL HMM (date={current_date.strftime('%Y-%m-%d')}):")
        print(f"    Training samples: {len(X_train_vol_scaled)}")
        
        # CONVERGENCE FIX: Enhanced fitting with robustness
        top_model, top_score, top_converged = fit_hmm_with_multiple_inits_robust(
            X_train_vol_scaled, 
            n_top_regimes, 
            HMM_TOP_N_INIT,
            HMM_TOP_N_ITER,
            HMM_TOP_TOL,
            HMM_TOP_RANDOM_STATE,
            HMM_TOP_COVARIANCE_TYPE,
            HMM_TOP_MIN_COVAR,
            feature_names=list(train_data_vol.columns)
        )
        
        if top_model is None:
            print("    TOP-LEVEL HMM: CONVERGENCE FAILED")
            return None, None, None, None, None
        
        # Predict top-level states for training data
        top_train_states = top_model.predict(X_train_vol_scaled)
        
        # Map HMM states to regimes by mean volatility
        regime_means = []
        for i in range(n_top_regimes):
            state_mask = top_train_states == i
            if state_mask.sum() > 0:
                mean_vol = train_data_vol.loc[train_data_vol.index[state_mask], 'RV_D'].mean()
                regime_means.append((i, mean_vol))
        
        regime_means_sorted = sorted(regime_means, key=lambda x: x[1])
        state_to_top_regime = {old: new for new, (old, _) in enumerate(regime_means_sorted)}
        
        # Predict top-level regime for current_date
        if current_date not in vol_features.index:
            return None, None, None, top_model, None
        
        X_current_vol = vol_features.loc[current_date:current_date].values
        X_current_vol_scaled = scaler_vol.transform(X_current_vol)
        top_state = top_model.predict(X_current_vol_scaled)[0]
        top_regime = state_to_top_regime[top_state]
        
        print(f"    TOP REGIME DETECTED: {['Low_Vol', 'Normal_Vol', 'High_Vol'][top_regime]}")
        
        # ====================================================================
        # LEVEL 2: SUB-LEVEL HMM (Micro sub-regime within macro regime)
        # ====================================================================
        
        # Filter to same top-level regime
        same_regime_mask = np.array([state_to_top_regime[s] == top_regime for s in top_train_states])
        
        # CONVERGENCE FIX: Stricter minimum data requirement
        if same_regime_mask.sum() < MIN_SUB_REGIME_DAYS:
            print(f"    SUB-HMM: Insufficient regime-specific data ({same_regime_mask.sum()} < {MIN_SUB_REGIME_DAYS})")
            return top_regime, None, f"{['Low_Vol', 'Normal_Vol', 'High_Vol'][top_regime]}_Insufficient", top_model, None
        
        # Get return features for same regime
        regime_specific_dates = train_data_ret.index[same_regime_mask]
        X_train_ret_regime = train_data_ret.loc[regime_specific_dates].values
        
        print(f"\n  SUB-LEVEL HMM (within {['Low_Vol', 'Normal_Vol', 'High_Vol'][top_regime]}):")
        print(f"    Training samples: {len(X_train_ret_regime)}")
        
        # CONVERGENCE FIX: RobustScaler for return features
        scaler_ret = RobustScaler()
        X_train_ret_scaled = scaler_ret.fit_transform(X_train_ret_regime)
        
        # CONVERGENCE FIX: Enhanced fitting for sub-HMM
        sub_model, sub_score, sub_converged = fit_hmm_with_multiple_inits_robust(
            X_train_ret_scaled,
            n_sub_regimes,
            HMM_SUB_N_INIT,
            HMM_SUB_N_ITER,
            HMM_SUB_TOL,
            HMM_SUB_RANDOM_STATE,
            HMM_SUB_COVARIANCE_TYPE,
            HMM_SUB_MIN_COVAR,
            feature_names=list(train_data_ret.columns)
        )
        
        if sub_model is None:
            print("    SUB-HMM: CONVERGENCE FAILED - using top-level only")
            return top_regime, None, f"{['Low_Vol', 'Normal_Vol', 'High_Vol'][top_regime]}_NoSub", top_model, None
        
        # Predict sub-level states
        sub_train_states = sub_model.predict(X_train_ret_scaled)
        
        # Map sub-states by mean returns
        sub_means = []
        for i in range(n_sub_regimes):
            sub_state_mask = sub_train_states == i
            if sub_state_mask.sum() > 0:
                mean_ret = train_data_ret.loc[regime_specific_dates[sub_state_mask], 'Ret_W'].mean()
                sub_means.append((i, mean_ret))
        
        sub_means_sorted = sorted(sub_means, key=lambda x: x[1])
        state_to_sub_regime = {old: new for new, (old, _) in enumerate(sub_means_sorted)}
        
        # Predict sub-regime for current_date
        if current_date not in ret_features.index:
            return top_regime, None, f"{['Low_Vol', 'Normal_Vol', 'High_Vol'][top_regime]}_NoDate", top_model, sub_model
        
        X_current_ret = ret_features.loc[current_date:current_date].values
        X_current_ret_scaled = scaler_ret.transform(X_current_ret)
        sub_state = sub_model.predict(X_current_ret_scaled)[0]
        sub_regime = state_to_sub_regime[sub_state]
        
        # ====================================================================
        # COMBINE REGIMES
        # ====================================================================
        
        top_names = ['Low_Vol', 'Normal_Vol', 'High_Vol']
        
        if top_regime == 0:  # Low Vol
            sub_names = ['Value', 'Growth']
        elif top_regime == 1:  # Normal Vol
            sub_names = ['Balanced', 'Rotation']
        else:  # High Vol
            sub_names = ['Defensive', 'FlightToSafety']
        
        combined_regime = f"{top_names[top_regime]}_{sub_names[sub_regime]}"
        
        print(f"    COMBINED REGIME: {combined_regime}")
        
        return top_regime, sub_regime, combined_regime, top_model, sub_model
        
    except Exception as e:
        print(f"  HHMM ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None, None, None


# ============================================================================
# PORTFOLIO CONSTRUCTION (UNCHANGED - SAME AS BEFORE)
# ============================================================================

def get_quasi_diag(link):
    """Reorder assets based on hierarchical tree clustering for HRP."""
    link = link.astype(int)
    sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
    num_items = link[-1, 3]
    
    while sort_ix.max() >= num_items:
        sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
        df0 = sort_ix[sort_ix >= num_items]
        i = df0.index
        j = df0.values - num_items
        sort_ix[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        sort_ix = pd.concat([sort_ix, df0])
        sort_ix = sort_ix.sort_index()
        sort_ix.index = range(sort_ix.shape[0])
    
    return sort_ix.tolist()


def get_cluster_var(cov, c_items):
    """Compute variance of a cluster using inverse-variance weights."""
    cov_slice = cov.loc[c_items, c_items]
    inv_diag = 1 / np.diag(cov_slice)
    w = inv_diag / inv_diag.sum()
    c_var = np.dot(w, np.dot(cov_slice, w))
    return c_var


def get_rec_bipart(cov, sort_ix):
    """Recursive bisection to compute HRP weights."""
    w = pd.Series(1, index=sort_ix)
    c_items = [sort_ix]
    
    while len(c_items) > 0:
        c_items = [i[j:k] for i in c_items 
                   for j, k in ((0, len(i) // 2), (len(i) // 2, len(i))) 
                   if len(i) > 1]
        
        for i in range(0, len(c_items), 2):
            c_items0 = c_items[i]
            c_items1 = c_items[i + 1]
            
            c_var0 = get_cluster_var(cov, c_items0)
            c_var1 = get_cluster_var(cov, c_items1)
            
            alpha = 1 - c_var0 / (c_var0 + c_var1)
            
            w[c_items0] *= alpha
            w[c_items1] *= 1 - alpha
    
    return w


def hrp_portfolio_canonical(returns, max_assets=None):
    """Canonical Hierarchical Risk Parity portfolio."""
    if len(returns) < 60:
        return {}
    
    try:
        corr = returns.corr()
        dist = np.sqrt((1 - corr) / 2)
        dist_condensed = squareform(dist.values, checks=False)
        link = linkage(dist_condensed, method='single')
        
        sort_ix = get_quasi_diag(link)
        sort_ix = [returns.columns[i] for i in sort_ix]
        
        cov = returns.cov() * 252
        weights = get_rec_bipart(cov, sort_ix)
        
        if max_assets is not None and len(weights) > max_assets:
            weights = weights.nlargest(max_assets)
            weights = weights / weights.sum()
        
        return {asset: w for asset, w in weights.items() if w > 0.01}
    
    except Exception as e:
        print(f"  HRP error: {str(e)}")
        return {}


def risk_parity_with_short_portfolio(returns, defensive_assets, short_assets, 
                                     min_weight=MIN_RISK_PARITY_WEIGHT, 
                                     max_weight=MAX_RISK_PARITY_WEIGHT):
    """Risk Parity Portfolio with SHORT EXPOSURE."""
    high_vol_universe = defensive_assets + short_assets
    available_assets = [a for a in high_vol_universe if a in returns.columns]
    
    if len(available_assets) < 2:
        return {asset: 1.0 / len(available_assets) for asset in available_assets}
    
    returns_subset = returns[available_assets]
    
    if len(returns_subset) < 60:
        return {asset: 1.0 / len(available_assets) for asset in available_assets}
    
    try:
        lw = LedoitWolf()
        cov_matrix = lw.fit(returns_subset.values).covariance_ * 252
        cov_matrix = pd.DataFrame(cov_matrix, index=available_assets, columns=available_assets)
        
        n_assets = len(available_assets)
        
        def risk_parity_objective(weights):
            port_var = np.dot(weights, np.dot(cov_matrix, weights))
            port_vol = np.sqrt(port_var)
            
            if port_vol < 1e-8:
                return 1e10
            
            marginal_contrib = np.dot(cov_matrix, weights)
            risk_contrib = weights * marginal_contrib / port_vol
            target_rc = port_vol / n_assets
            
            return np.sum((risk_contrib - target_rc) ** 2)
        
        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0}]
        bounds = [(min_weight, max_weight) for _ in range(n_assets)]
        x0 = np.array([1.0 / n_assets] * n_assets)
        
        result = minimize(
            risk_parity_objective,
            x0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'maxiter': 1000, 'ftol': 1e-9}
        )
        
        if not result.success:
            return {asset: 1.0 / n_assets for asset in available_assets}
        
        weights = result.x
        weights_dict = {asset: w for asset, w in zip(available_assets, weights) if w > 0.01}
        total_weight = sum(weights_dict.values())
        if total_weight > 0:
            weights_dict = {k: v / total_weight for k, v in weights_dict.items()}
        
        return weights_dict
    
    except Exception as e:
        print(f"  Risk Parity error: {str(e)}")
        return {asset: 1.0 / len(available_assets) for asset in available_assets}


def max_sharpe_portfolio(returns, risk_free_rate=0.02):
    """Maximum Sharpe Ratio portfolio optimization."""
    if len(returns) < 60:
        return {}
    
    mu = returns.mean() * 252
    
    try:
        lw = LedoitWolf()
        cov_matrix = lw.fit(returns.values).covariance_ * 252
    except:
        return {}
    
    n_assets = len(mu)
    
    def neg_sharpe(weights):
        port_return = np.dot(weights, mu)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return -(port_return - risk_free_rate) / port_vol if port_vol > 0 else -np.inf
    
    constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0}]
    bounds = [(0, 0.3) for _ in range(n_assets)]
    x0 = np.array([1.0 / n_assets] * n_assets)
    
    result = minimize(neg_sharpe, x0, method='SLSQP', bounds=bounds, 
                     constraints=constraints, options={'maxiter': 1000})
    
    if result.success:
        weights = result.x
        weights = np.where(weights < 0.01, 0, weights)
        if np.sum(weights) > 0:
            weights = weights / np.sum(weights)
            return {asset: w for asset, w in zip(returns.columns, weights) if w > 0.01}
    
    return {}


# ============================================================================
# BENCHMARK & BACKTEST (UNCHANGED FROM ORIGINAL)
# ============================================================================

def create_buy_hold_benchmark(benchmark_data, regime_dates, initial_capital, transaction_cost):
    """Create Buy & Hold benchmark strategy."""
    print(f"\n{'='*80}")
    print("CREATING BUY & HOLD BENCHMARK")
    print(f"{'='*80}")
    
    benchmark_aligned = benchmark_data.reindex(regime_dates)
    
    first_valid_idx = benchmark_aligned['close'].first_valid_index()
    if first_valid_idx is None:
        print("  ✗ No valid benchmark data")
        return None
    
    first_price = benchmark_aligned.loc[first_valid_idx, 'close']
    net_capital = initial_capital * (1 - transaction_cost)
    shares = net_capital / first_price
    
    print(f"  Initial purchase: {shares:.2f} shares at ${first_price:.2f}")
    print(f"  Transaction cost: ${initial_capital * transaction_cost:.2f}")
    
    benchmark_values = []
    for date in regime_dates:
        if date >= first_valid_idx and not pd.isna(benchmark_aligned.loc[date, 'close']):
            current_price = benchmark_aligned.loc[date, 'close']
            portfolio_value = shares * current_price
            
            benchmark_values.append({
                'date': date,
                'value': portfolio_value,
                'price': current_price
            })
    
    benchmark_df = pd.DataFrame(benchmark_values)
    benchmark_df['return'] = benchmark_df['value'].pct_change()
    
    print(f"  ✓ Benchmark created: {len(benchmark_df)} days")
    print(f"  Final value: ${benchmark_df['value'].iloc[-1]:,.2f}")
    
    return benchmark_df


class HierarchicalRegimePortfolioBacktest:
    """
    Backtest engine with HIERARCHICAL HMM regime detection.
    UNCHANGED - same implementation as before.
    """
    
    def __init__(self, vol_features, ret_features, asset_data, initial_capital, transaction_cost):
        self.vol_features = vol_features
        self.ret_features = ret_features
        self.asset_data = asset_data
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost
        
        self.capital = initial_capital
        self.cash = initial_capital
        self.positions = {}
        self.portfolio_history = []
        self.trades = []
        self.regime_history = []
        
        self.align_asset_data()
    
    def align_asset_data(self):
        """Align all asset data to common dates."""
        common_dates = self.vol_features.index
        for ticker in self.asset_data:
            self.asset_data[ticker] = self.asset_data[ticker].reindex(common_dates)
    
    def get_current_prices(self, date, price_type='close'):
        """Get current prices for all assets."""
        prices = {}
        for ticker, df in self.asset_data.items():
            if date in df.index and not pd.isna(df.loc[date, price_type]):
                prices[ticker] = df.loc[date, price_type]
        return prices
    
    def calculate_portfolio_value(self, date):
        """Calculate total portfolio value at given date."""
        prices = self.get_current_prices(date, 'close')
        risky_value = sum(self.positions.get(ticker, 0) * prices.get(ticker, 0) 
                         for ticker in self.positions)
        return self.cash + risky_value
    
    def execute_rebalance(self, date, target_weights):
        """Execute portfolio rebalancing."""
        dates_list = list(self.vol_features.index)
        current_idx = dates_list.index(date)
        
        if current_idx >= len(dates_list) - 1:
            return
        
        next_date = dates_list[current_idx + 1]
        exec_prices = self.get_current_prices(next_date, 'open')
        
        if not exec_prices:
            return
        
        portfolio_value = self.calculate_portfolio_value(date)
        
        target_positions = {}
        for ticker, weight in target_weights.items():
            if ticker in exec_prices:
                target_value = portfolio_value * weight
                target_positions[ticker] = target_value / exec_prices[ticker]
        
        # Close unwanted positions
        for ticker in list(self.positions.keys()):
            if ticker not in target_positions and self.positions[ticker] != 0:
                if ticker in exec_prices:
                    sell_value = self.positions[ticker] * exec_prices[ticker]
                    cost = abs(sell_value) * self.transaction_cost
                    self.cash += sell_value - cost
                    self.positions[ticker] = 0
                    
                    self.trades.append({
                        'date': next_date,
                        'ticker': ticker,
                        'action': 'SELL',
                        'value': sell_value,
                        'cost': cost
                    })
        
        # Adjust to target positions
        for ticker, target_shares in target_positions.items():
            current_shares = self.positions.get(ticker, 0)
            delta_shares = target_shares - current_shares
            
            if abs(delta_shares) > 0.01:
                trade_value = delta_shares * exec_prices[ticker]
                cost = abs(trade_value) * self.transaction_cost
                
                if trade_value > 0:
                    if self.cash >= (trade_value + cost):
                        self.positions[ticker] = target_shares
                        self.cash -= (trade_value + cost)
                        
                        self.trades.append({
                            'date': next_date,
                            'ticker': ticker,
                            'action': 'BUY',
                            'value': trade_value,
                            'cost': cost
                        })
                
                else:
                    self.positions[ticker] = target_shares
                    self.cash += (-trade_value - cost)
                    
                    self.trades.append({
                        'date': next_date,
                        'ticker': ticker,
                        'action': 'SELL',
                        'value': trade_value,
                        'cost': cost
                    })
    
    def run_backtest(self):
        """Execute full backtest with HIERARCHICAL HMM regime detection."""
        
        print(f"\n{'='*80}")
        print("RUNNING HIERARCHICAL HMM BACKTEST (WITH CONVERGENCE FIXES)")
        print(f"{'='*80}")
        
        dates = list(self.vol_features.index)
        last_rebalance = -REGIME_REBALANCE_FREQ
        
        for day_num, date in enumerate(dates):
            # Hierarchical regime detection
            top_regime, sub_regime, combined_regime, top_model, sub_model = \
                detect_hierarchical_regime_walk_forward(
                    self.vol_features, self.ret_features, date, 
                    N_TOP_REGIMES, N_SUB_REGIMES, MIN_TRAIN_DAYS
                )
            
            if top_regime is None or sub_regime is None:
                portfolio_value = self.calculate_portfolio_value(date)
                self.portfolio_history.append({
                    'date': date,
                    'value': portfolio_value,
                    'cash': self.cash,
                    'top_regime': None,
                    'sub_regime': None,
                    'combined_regime': 'Insufficient_Data',
                    'num_positions': len([p for p in self.positions.values() if p != 0])
                })
                continue
            
            self.regime_history.append({
                'date': date,
                'top_regime': top_regime,
                'sub_regime': sub_regime,
                'combined_regime': combined_regime
            })
            
            portfolio_value = self.calculate_portfolio_value(date)
            
            # Rebalancing logic
            days_since_rebal = day_num - last_rebalance
            
            if days_since_rebal >= REGIME_REBALANCE_FREQ and day_num >= MIN_TRAIN_DAYS + 60:
                hist_start = max(0, day_num - 252)
                hist_dates = dates[hist_start:day_num]
                
                returns_list = []
                for ticker in self.asset_data:
                    ticker_prices = self.asset_data[ticker].loc[hist_dates, 'close']
                    ticker_returns = ticker_prices.pct_change().dropna()
                    returns_list.append(ticker_returns.rename(ticker))
                
                if returns_list:
                    returns_df = pd.concat(returns_list, axis=1).dropna()
                    
                    # Portfolio allocation by regime (SAME AS BEFORE)
                    if top_regime == 0 and sub_regime == 0:
                        value_assets = ['XLV', 'XLF', 'VTI', 'SPY']
                        value_returns = returns_df[[c for c in value_assets if c in returns_df.columns]]
                        target_weights = max_sharpe_portfolio(value_returns)
                        strategy_name = "Low_Vol_Value"
                    
                    elif top_regime == 0 and sub_regime == 1:
                        growth_assets = ['QQQ', 'XLK', 'AAPL', 'MSFT', 'VTI']
                        growth_returns = returns_df[[c for c in growth_assets if c in returns_df.columns]]
                        target_weights = max_sharpe_portfolio(growth_returns)
                        strategy_name = "Low_Vol_Growth"
                    
                    elif top_regime == 1 and sub_regime == 0:
                        normal_assets = GROWTH_ASSETS + DEFENSIVE_ASSETS
                        balanced_returns = returns_df[[c for c in normal_assets if c in returns_df.columns]]
                        target_weights = hrp_portfolio_canonical(balanced_returns, max_assets=MAX_HRP_ASSETS)
                        strategy_name = "Normal_Vol_Balanced"
                    
                    elif top_regime == 1 and sub_regime == 1:
                        momentum_scores = {}
                        for ticker in GROWTH_ASSETS + DEFENSIVE_ASSETS:
                            if ticker in returns_df.columns:
                                momentum_scores[ticker] = returns_df[ticker].iloc[-21:].sum()
                        
                        top_momentum = sorted(momentum_scores.items(), key=lambda x: x[1], reverse=True)[:8]
                        rotation_assets = [t[0] for t in top_momentum]
                        rotation_returns = returns_df[rotation_assets]
                        target_weights = hrp_portfolio_canonical(rotation_returns, max_assets=8)
                        strategy_name = "Normal_Vol_Rotation"
                    
                    elif top_regime == 2 and sub_regime == 0:
                        target_weights = risk_parity_with_short_portfolio(
                            returns_df,
                            defensive_assets=DEFENSIVE_ASSETS,
                            short_assets=SHORT_ASSETS,
                            min_weight=0.10,
                            max_weight=0.35
                        )
                        strategy_name = "High_Vol_Defensive"
                    
                    else:
                        target_weights = risk_parity_with_short_portfolio(
                            returns_df,
                            defensive_assets=DEFENSIVE_ASSETS,
                            short_assets=SHORT_ASSETS,
                            min_weight=0.15,
                            max_weight=0.60
                        )
                        strategy_name = "High_Vol_FlightToSafety"
                    
                    if target_weights:
                        print(f"\n  Day {day_num}: {date.strftime('%Y-%m-%d')} - {combined_regime}")
                        print(f"    Strategy: {strategy_name}")
                        print(f"    Assets: {list(target_weights.keys())}")
                        
                        self.execute_rebalance(date, target_weights)
                        last_rebalance = day_num
            
            self.portfolio_history.append({
                'date': date,
                'value': portfolio_value,
                'cash': self.cash,
                'top_regime': top_regime,
                'sub_regime': sub_regime,
                'combined_regime': combined_regime,
                'num_positions': len([p for p in self.positions.values() if p != 0])
            })
            
            if day_num % 100 == 0:
                print(f"  Progress: {day_num}/{len(dates)} days ({(day_num/len(dates)*100):.1f}%)")
        
        print(f"\n✓ Backtest complete")
        return self.portfolio_history, self.trades


# ============================================================================
# PERFORMANCE & VISUALIZATION (UNCHANGED)
# ============================================================================

def calculate_performance_metrics(portfolio_df, initial_capital):
    """Calculate comprehensive performance metrics."""
    final_value = portfolio_df['value'].iloc[-1]
    total_return = (final_value / initial_capital - 1) * 100
    
    returns = portfolio_df['value'].pct_change().dropna()
    
    annual_return = returns.mean() * 252 * 100
    annual_vol = returns.std() * np.sqrt(252) * 100
    sharpe = (annual_return - 2.0) / annual_vol if annual_vol > 0 else 0
    
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min() * 100
    
    calmar = annual_return / abs(max_dd) if max_dd != 0 else 0
    win_rate = (returns > 0).sum() / len(returns) * 100 if len(returns) > 0 else 0
    
    return {
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_vol': annual_vol,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'calmar': calmar,
        'win_rate': win_rate,
        'final_value': final_value
    }


def print_comparative_metrics(strategy_metrics, benchmark_metrics):
    """Print side-by-side comparison."""
    print(f"\n{'='*80}")
    print("COMPARATIVE PERFORMANCE: HIERARCHICAL HMM vs BENCHMARK")
    print(f"{'='*80}")
    print(f"\n{'Metric':<25} {'Strategy':>15} {'Benchmark':>15} {'Difference':>15}")
    print("-" * 80)
    
    metrics_to_compare = [
        ('Final Value', 'final_value', '${:,.0f}'),
        ('Total Return', 'total_return', '{:.2f}%'),
        ('Annual Return', 'annual_return', '{:.2f}%'),
        ('Annual Volatility', 'annual_vol', '{:.2f}%'),
        ('Sharpe Ratio', 'sharpe', '{:.3f}'),
        ('Maximum Drawdown', 'max_dd', '{:.2f}%'),
        ('Calmar Ratio', 'calmar', '{:.3f}'),
        ('Win Rate', 'win_rate', '{:.2f}%')
    ]
    
    for label, key, fmt in metrics_to_compare:
        strat_val = strategy_metrics[key]
        bench_val = benchmark_metrics[key]
        
        if key == 'final_value':
            diff = strat_val - bench_val
            diff_str = f"${diff:,.0f}"
        else:
            diff = strat_val - bench_val
            diff_str = f"{diff:+.2f}"
        
        print(f"{label:<25} {fmt.format(strat_val):>15} {fmt.format(bench_val):>15} {diff_str:>15}")
    
    print("-" * 80)


def plot_hierarchical_regime_portfolio(portfolio_history, benchmark_df):
    """Visualize hierarchical regime-conditional portfolio."""
    
    portfolio_df = pd.DataFrame(portfolio_history)
    portfolio_df = portfolio_df[portfolio_df['top_regime'].notna()]
    
    fig, axes = plt.subplots(6, 1, figsize=(16, 22))
    
    regime_colors = {
        'Low_Vol_Value': '#90EE90',
        'Low_Vol_Growth': '#00FF00',
        'Normal_Vol_Balanced': '#FFD700',
        'Normal_Vol_Rotation': '#FFA500',
        'High_Vol_Defensive': '#FF6B6B',
        'High_Vol_FlightToSafety': '#DC143C'
    }
    
    # Plot 1: Portfolio Value
    for regime_name, color in regime_colors.items():
        mask = portfolio_df['combined_regime'] == regime_name
        if mask.sum() > 0:
            axes[0].scatter(portfolio_df.loc[mask, 'date'],
                           portfolio_df.loc[mask, 'value'],
                           c=color,
                           s=10,
                           alpha=0.6,
                           label=f"{regime_name}")
    
    axes[0].plot(benchmark_df['date'], benchmark_df['value'],
                color='black', linewidth=2, linestyle='--',
                label='Benchmark (URTH)', alpha=0.7)
    
    axes[0].set_ylabel('Portfolio Value ($)', fontsize=11)
    axes[0].set_title('Hierarchical HMM Strategy vs Benchmark (WITH CONVERGENCE FIXES)', 
                     fontsize=14, fontweight='bold')
    axes[0].legend(loc='upper left', fontsize=8, ncol=2)
    axes[0].grid(alpha=0.3)
    
    # Plot 2: Top-level regime
    top_regime_colors = {0: '#90EE90', 1: '#FFD700', 2: '#FF6B6B'}
    
    axes[1].scatter(portfolio_df['date'],
                   portfolio_df['top_regime'],
                   c=[top_regime_colors[r] for r in portfolio_df['top_regime']],
                   s=20,
                   alpha=0.6)
    axes[1].set_ylabel('Top Regime', fontsize=11)
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_yticklabels(['Low Vol', 'Normal Vol', 'High Vol'])
    axes[1].set_title('Top-Level Regime', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    # Plot 3: Sub-level regime
    axes[2].scatter(portfolio_df['date'],
                   portfolio_df['sub_regime'],
                   c='purple',
                   s=20,
                   alpha=0.5)
    axes[2].set_ylabel('Sub Regime', fontsize=11)
    axes[2].set_yticks([0, 1])
    axes[2].set_yticklabels(['Sub-0', 'Sub-1'])
    axes[2].set_title('Sub-Level Regime', fontsize=14, fontweight='bold')
    axes[2].grid(alpha=0.3)
    
    # Plot 4: Cumulative returns
    portfolio_df['return'] = portfolio_df['value'].pct_change()
    strategy_cum_ret = portfolio_df['return'].cumsum() * 100
    benchmark_cum_ret = benchmark_df['return'].cumsum() * 100
    
    axes[3].plot(portfolio_df['date'], strategy_cum_ret, 
                color='blue', linewidth=2, label='Strategy', alpha=0.8)
    axes[3].plot(benchmark_df['date'], benchmark_cum_ret,
                color='black', linewidth=2, linestyle='--', label='Benchmark', alpha=0.7)
    axes[3].set_ylabel('Cumulative Return (%)', fontsize=11)
    axes[3].set_title('Cumulative Returns', fontsize=14, fontweight='bold')
    axes[3].legend()
    axes[3].grid(alpha=0.3)
    axes[3].axhline(y=0, color='black', linestyle='--', linewidth=1)
    
    # Plot 5: Rolling Sharpe
    strategy_rolling_sharpe = (
        portfolio_df['return'].rolling(63).mean() * 252 / 
        (portfolio_df['return'].rolling(63).std() * np.sqrt(252))
    )
    
    benchmark_rolling_sharpe = (
        benchmark_df['return'].rolling(63).mean() * 252 / 
        (benchmark_df['return'].rolling(63).std() * np.sqrt(252))
    )
    
    axes[4].plot(portfolio_df['date'], strategy_rolling_sharpe,
                color='blue', linewidth=1.5, label='Strategy', alpha=0.8)
    axes[4].plot(benchmark_df['date'], benchmark_rolling_sharpe,
                color='black', linewidth=1.5, linestyle='--', label='Benchmark', alpha=0.7)
    axes[4].set_ylabel('Rolling Sharpe (63d)', fontsize=11)
    axes[4].set_title('Rolling Sharpe Ratio', fontsize=14, fontweight='bold')
    axes[4].axhline(y=0, color='red', linestyle='--', linewidth=1)
    axes[4].legend()
    axes[4].grid(alpha=0.3)
    
    # Plot 6: Drawdown
    strategy_returns = portfolio_df['return'].dropna()
    strategy_cumulative = (1 + strategy_returns).cumprod()
    strategy_running_max = strategy_cumulative.expanding().max()
    strategy_drawdown = (strategy_cumulative - strategy_running_max) / strategy_running_max
    
    benchmark_returns = benchmark_df['return'].dropna()
    benchmark_cumulative = (1 + benchmark_returns).cumprod()
    benchmark_running_max = benchmark_cumulative.expanding().max()
    benchmark_drawdown = (benchmark_cumulative - benchmark_running_max) / benchmark_running_max
    
    axes[5].fill_between(portfolio_df['date'].iloc[1:], strategy_drawdown.values * 100, 0,
                        color='blue', alpha=0.5, label='Strategy')
    axes[5].fill_between(benchmark_df['date'].iloc[1:], benchmark_drawdown.values * 100, 0,
                        color='black', alpha=0.3, label='Benchmark')
    axes[5].set_ylabel('Drawdown (%)', fontsize=11)
    axes[5].set_xlabel('Date', fontsize=11)
    axes[5].set_title('Drawdown Comparison', fontsize=14, fontweight='bold')
    axes[5].legend()
    axes[5].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('hierarchical_hmm_convergence_fixed.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Visualization saved as 'hierarchical_hmm_convergence_fixed.png'")
    plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Execute HIERARCHICAL HMM with CONVERGENCE FIXES APPLIED.
    """
    
    print("="*80)
    print("HIERARCHICAL HMM - WITH CONVERGENCE FIXES")
    print("="*80)
    print("\nCONVERGENCE IMPROVEMENTS APPLIED:")
    print("  1. Increased minimum covariance (1e-3 vs 1e-4)")
    print("  2. More HMM iterations (300 vs 200 top, 250 vs 150 sub)")
    print("  3. Relaxed convergence tolerance (5e-3 vs 1e-3)")
    print("  4. More initialization attempts (10 vs 5)")
    print("  5. RobustScaler instead of StandardScaler")
    print("  6. Outlier removal and data validation")
    print("  7. Stronger regime persistence in transition matrix")
    print("  8. Increased minimum sub-regime data (90 vs 60 days)")
    print("  9. Better K-means initialization (20 vs 10 restarts)")
    print(" 10. Comprehensive error handling and fallbacks")
    print("="*80)
    
    try:
        # Data acquisition
        print(f"\nFetching {REGIME_TICKER}...")
        spy_data = fetch_price_data(REGIME_TICKER, START_DATE, END_DATE)
        
        vix_data = None
        if USE_VIX:
            print("Fetching VIX...")
            try:
                vix_raw = fetch_price_data('%5EVIX', START_DATE, END_DATE)
                if vix_raw is not None:
                    vix_data = vix_raw['close']
            except:
                print("  Warning: Could not fetch VIX")
        
        print(f"\nFetching {BENCHMARK_TICKER}...")
        benchmark_data = fetch_price_data(BENCHMARK_TICKER, START_DATE, END_DATE)
        
        asset_data = fetch_all_assets(ALL_ASSETS, START_DATE, END_DATE)
        
        # Feature preparation
        vol_features, ret_features = prepare_regime_features(spy_data, vix_data)
        
        # Backtest
        backtest = HierarchicalRegimePortfolioBacktest(
            vol_features=vol_features,
            ret_features=ret_features,
            asset_data=asset_data,
            initial_capital=INITIAL_CAPITAL,
            transaction_cost=TRANSACTION_COST
        )
        
        portfolio_history, trades = backtest.run_backtest()
        
        # Benchmark
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df = portfolio_df[portfolio_df['top_regime'].notna()]
        
        benchmark_df = None
        if benchmark_data is not None:
            benchmark_df = create_buy_hold_benchmark(
                benchmark_data=benchmark_data,
                regime_dates=portfolio_df['date'].tolist(),
                initial_capital=INITIAL_CAPITAL,
                transaction_cost=TRANSACTION_COST
            )
        
        # Performance
        print(f"\n{'='*80}")
        print("PERFORMANCE SUMMARY (WITH CONVERGENCE FIXES)")
        print(f"{'='*80}")
        
        strategy_metrics = calculate_performance_metrics(portfolio_df, INITIAL_CAPITAL)
        
        print(f"\nStrategy Performance:")
        print(f"  Initial Capital:   ${INITIAL_CAPITAL:,.0f}")
        print(f"  Final Value:       ${strategy_metrics['final_value']:,.0f}")
        print(f"  Total Return:      {strategy_metrics['total_return']:.2f}%")
        print(f"  Annual Return:     {strategy_metrics['annual_return']:.2f}%")
        print(f"  Sharpe Ratio:      {strategy_metrics['sharpe']:.3f}")
        print(f"  Maximum Drawdown:  {strategy_metrics['max_dd']:.2f}%")
        print(f"  Total Trades:      {len(trades)}")
        
        if benchmark_df is not None:
            benchmark_metrics = calculate_performance_metrics(benchmark_df, INITIAL_CAPITAL)
            print_comparative_metrics(strategy_metrics, benchmark_metrics)
        
        # Regime statistics
        print(f"\n{'='*80}")
        print("REGIME STATISTICS")
        print(f"{'='*80}")
        
        regime_counts = portfolio_df['combined_regime'].value_counts()
        
        for regime_name in regime_counts.index:
            if 'Insufficient' not in regime_name:
                regime_mask = portfolio_df['combined_regime'] == regime_name
                n_days = regime_mask.sum()
                pct = n_days / len(portfolio_df) * 100
                
                regime_returns = portfolio_df.loc[regime_mask, 'value'].pct_change().dropna()
                regime_sharpe = (regime_returns.mean() * 252) / (regime_returns.std() * np.sqrt(252)) if len(regime_returns) > 0 else 0
                
                print(f"\n{regime_name}:")
                print(f"  Days:         {n_days} ({pct:.1f}%)")
                print(f"  Sharpe Ratio: {regime_sharpe:.3f}")
        
        # Visualization
        if benchmark_df is not None:
            plot_hierarchical_regime_portfolio(portfolio_history, benchmark_df)
        
        print(f"\n{'='*80}")
        print("HIERARCHICAL HMM EXECUTION COMPLETE (CONVERGENCE FIXES APPLIED)")
        print(f"{'='*80}")
        
        return portfolio_history, trades, backtest.regime_history
    
    except Exception as e:
        print(f"\nERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None


if __name__ == "__main__":
    portfolio_history, trades, regime_history = main()